<a href="https://colab.research.google.com/github/onlyrituraj/Slide-to-PDF/blob/main/Slide_to_Handout.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📄 Slides → Handout Converter
**Converts any PDF (presentation slides) into a printable 3-slides-per-page handout.**

Steps:
1. Run Cell 1 → installs library
2. Run Cell 2 → upload your PDF
3. Run Cell 3 → converts & downloads the handout

In [1]:
# ── Cell 1: Install dependency ──────────────────────────────────────────────
!pip install pymupdf -q
print('✅ pymupdf installed!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 69.9 MB/s eta 0:00:00
✅ pymupdf installed!


In [2]:
# ── Cell 2: Upload your PDF ─────────────────────────────────────────────────
from google.colab import files

print('📂 Select your PDF file to upload...')
uploaded = files.upload()

input_filename = list(uploaded.keys())[0]
print(f'\n✅ Uploaded: {input_filename}')

📂 Select your PDF file to upload...


Saving OS_1pptx__1_pdf_lyst1752818659470.pdf to OS_1pptx__1_pdf_lyst1752818659470.pdf

✅ Uploaded: OS_1pptx__1_pdf_lyst1752818659470.pdf


In [3]:
# ── Cell 3: Convert & Download ──────────────────────────────────────────────
import pymupdf
import os
from google.colab import files

# ── Settings (tweak if needed) ───────────────────────────────────────────────
SLIDES_PER_PAGE = 3       # slides per output page (try 2 or 4)
RENDER_DPI      = 150     # 150 = good screen quality | 200-300 = print quality
MARGIN_PT       = 20      # outer page margin in points
SLIDE_GAP_PT    = 12      # gap between slides
BORDER_PT       = 1       # thin border around each slide (0 = off)
PAGE_W          = 595.28  # A4 width  in points
PAGE_H          = 841.89  # A4 height in points
# ─────────────────────────────────────────────────────────────────────────────

def get_thumb_rect(slot_idx, pix_w, pix_h):
    """Return (Rect) for the slot_idx-th thumbnail on the output page."""
    available_h = PAGE_H - 2 * MARGIN_PT - (SLIDES_PER_PAGE - 1) * SLIDE_GAP_PT
    slot_h      = available_h / SLIDES_PER_PAGE
    slot_w      = PAGE_W - 2 * MARGIN_PT

    scale = min(slot_w / pix_w, slot_h / pix_h)
    tw, th = pix_w * scale, pix_h * scale

    x0 = MARGIN_PT + (slot_w - tw) / 2
    y0 = MARGIN_PT + slot_idx * (slot_h + SLIDE_GAP_PT) + (slot_h - th) / 2
    return pymupdf.Rect(x0, y0, x0 + tw, y0 + th)


def convert(input_path, output_path):
    src = pymupdf.open(input_path)
    total = len(src)
    print(f'📖 Opened "{input_path}" — {total} slides')

    dst      = pymupdf.open()
    out_page = None
    slot_idx = 0

    for i in range(total):
        if slot_idx == 0:
            out_page = dst.new_page(width=PAGE_W, height=PAGE_H)
            out_page.draw_rect(out_page.rect, color=(1,1,1), fill=(1,1,1))

        slide = src[i]
        mat   = pymupdf.Matrix(RENDER_DPI / 72, RENDER_DPI / 72)
        pix   = slide.get_pixmap(matrix=mat, alpha=False, colorspace=pymupdf.csRGB)

        rect = get_thumb_rect(slot_idx, pix.width, pix.height)

        if BORDER_PT > 0:
            border = rect + (-BORDER_PT, -BORDER_PT, BORDER_PT, BORDER_PT)
            out_page.draw_rect(border, color=(0.6,0.6,0.6), fill=None, width=BORDER_PT)

        out_page.insert_image(rect, pixmap=pix)

        out_page_num = len(dst)
        print(f'  ✔ slide {i+1}/{total}  →  output page {out_page_num}, slot {slot_idx+1}')
        slot_idx = (slot_idx + 1) % SLIDES_PER_PAGE

    dst.save(output_path, garbage=4, deflate=True, clean=True)
    src.close(); dst.close()

    size_kb    = os.path.getsize(output_path) / 1024
    out_pages  = -(-total // SLIDES_PER_PAGE)   # ceiling division
    print(f'\n✅ Done! {total} slides → {out_pages} A4 pages ({size_kb:.0f} KB)')


# Run conversion
base, _       = os.path.splitext(input_filename)
output_filename = base + '_handout.pdf'

convert(input_filename, output_filename)

# Auto-download the result
print(f'\n⬇️  Downloading "{output_filename}"...')
files.download(output_filename)

📖 Opened "OS_1pptx__1_pdf_lyst1752818659470.pdf" — 518 slides
  ✔ slide 1/518  →  output page 1, slot 1
  ✔ slide 2/518  →  output page 1, slot 2
  ✔ slide 3/518  →  output page 1, slot 3
  ✔ slide 4/518  →  output page 2, slot 1
  ✔ slide 5/518  →  output page 2, slot 2
  ✔ slide 6/518  →  output page 2, slot 3
  ✔ slide 7/518  →  output page 3, slot 1
  ✔ slide 8/518  →  output page 3, slot 2
  ✔ slide 9/518  →  output page 3, slot 3
  ✔ slide 10/518  →  output page 4, slot 1
  ✔ slide 11/518  →  output page 4, slot 2
  ✔ slide 12/518  →  output page 4, slot 3
  ✔ slide 13/518  →  output page 5, slot 1
  ✔ slide 14/518  →  output page 5, slot 2
  ✔ slide 15/518  →  output page 5, slot 3
  ✔ slide 16/518  →  output page 6, slot 1
  ✔ slide 17/518  →  output page 6, slot 2
  ✔ slide 18/518  →  output page 6, slot 3
  ✔ slide 19/518  →  output page 7, slot 1
  ✔ slide 20/518  →  output page 7, slot 2
  ✔ slide 21/518  →  output page 7, slot 3
  ✔ slide 22/518  →  output page 8, slot 1
 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>